In [41]:
import pandas as pd

from utils import create_match_keys, clean_text_cols

In [78]:
arb_candidates = pd.read_json('data/arb_candidates.json')
arb_candidates.values[0]

array([{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'poly': '307120'},
       'salavat yulaev ufa vs avtomobilist yekaterinburg',
       'khl salavat yulaev ufa vs avtomobilist yekaterinburg',
       0.9490000000000001,
       {'kalshi': 'unknown_2026', 'poly': 'unknown_9999'}], dtype=object)

In [43]:
import requests
import pandas as pd

def hydrate_raw_arbitrage_data(df: pd.DataFrame):
    """
    Fetches raw market data for Poly and Kalshi and returns them as DataFrames.
    No field selection is performed; returns the full API response objects.
    """
    ids_list = df['platform_ids'].tolist()
    poly_ids = [d.get('poly') for d in ids_list if d.get('poly')]
    kalshi_tickers = [d.get('kalshi') for d in ids_list if d.get('kalshi')]

    poly_rows = []
    kalshi_rows = []

    # --- Polymarket (Raw Batch Dump) ---
    if poly_ids:
        try:
            url = "https://gamma-api.polymarket.com/events"
            params = [('id', eid) for eid in poly_ids]
            response = requests.get(url, params=params, timeout=10)
            
            if response.status_code == 200:
                for event in response.json():
                    event_id = str(event.get('id'))
                    for market in event.get('markets', []):
                        # Flattening slightly so we have the parent event ID
                        raw_market = market.copy()
                        raw_market['event_id'] = event_id
                        poly_rows.append(raw_market)
        except Exception as e:
            print(f"Batch Poly error: {e}")

    # --- Kalshi (Raw Individual Dump) ---
    for ticker in kalshi_tickers:
        try:
            url = f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}?with_nested_markets=true"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                event_data = response.json().get('event', {})
                parent_ticker = event_data.get('ticker')
                for market in event_data.get('markets', []):
                    raw_market = market.copy()
                    raw_market['event_id'] = ticker
                    kalshi_rows.append(raw_market)
        except Exception as e:
            print(f"Kalshi error for {ticker}: {e}")

    return pd.DataFrame(poly_rows), pd.DataFrame(kalshi_rows)

# Usage:
poly_raw_df, kalshi_raw_df = hydrate_raw_arbitrage_data(arb_candidates)

In [75]:

poly_raw_df.rename(columns={'id': 'market_id'}, inplace=True)
cols_to_keep = ['event_id', 'market_id', 'question', 'description', 'outcomes']

poly_df = poly_raw_df[cols_to_keep]

clean_outcomes = (
    poly_df['outcomes']
    .str.replace(r"[\[\]']", "", regex=True) # Removes [, ], and '
    .str.replace('"', "")                    # Removes "
)


poly_df['descriptive_text'] = (
    poly_df['question'].fillna('') + " " + 
    # poly_df['description'].fillna('') + " " + 
    clean_outcomes.fillna('')
).str.strip()

poly_df = clean_text_cols(poly_df)

poly_df = create_match_keys(poly_df, ['descriptive_text'])
# poly_df.drop(columns=['question', 'description', 'outcomes'], inplace=True)
poly_df.head()

,event_id,market_id,question,description,outcomes,descriptive_text,match_key
0,305272,1746692,boston red sox vs cincinnati reds o u 7 5,in the upcoming mlb game between the boston re...,over under,boston red sox vs cincinnati reds o u 7 5 over...,boston_cincinnati_9999
1,305272,1746162,spread boston red sox 1 5,in the upcoming mlb game between the boston re...,boston red sox cincinnati reds,spread boston red sox 1 5 boston red sox cinci...,boston_cincinnati_9999
2,305272,1746163,boston red sox vs cincinnati reds o u 8 5,in the upcoming mlb game between the boston re...,over under,boston red sox vs cincinnati reds o u 8 5 over...,boston_cincinnati_9999
3,305272,1711557,nrfi boston red sox vs cincinnati reds,in the upcoming mlb game between the boston re...,yes no,nrfi boston red sox vs cincinnati reds yes no,boston_cincinnati_yes_9999
4,305272,1711556,boston red sox vs cincinnati reds,in the upcoming mlb game between the boston re...,boston red sox cincinnati reds,boston red sox vs cincinnati reds boston red s...,boston_cincinnati_9999


In [66]:

cols_to_keep_kalshi = [
    'event_id', 'ticker', 'title', 
    # 'rules_primary', 'result', 
    # 'rules_secondary', 'yes_sub_title'
]

kalshi_df = kalshi_raw_df[cols_to_keep_kalshi].copy() # Use .copy() to avoid SettingWithCopyWarning

# Use this pattern instead
kalshi_df['descriptive_text'] = (
    # kalshi_df['rules_primary'].fillna('').astype(str) + " " + 
    # kalshi_df['result'].fillna('').astype(str) + " " + 
    # kalshi_df['rules_secondary'].fillna('').astype(str) + " " + 
    kalshi_df['title'].fillna('').astype(str) 
    # kalshi_df['yes_sub_title'].fillna('').astype(str)
).str.replace(r'\s+', ' ', regex=True).str.strip()

kalshi_df = clean_text_cols(kalshi_df)
kalshi_df = create_match_keys(kalshi_df, ['descriptive_text'])
# kalshi_df.drop(columns=['rules_primary', 'result', 'rules_secondary', 'title', 'yes_sub_title'], inplace=True)
kalshi_df.rename(columns={'ticker': 'market_id'}, inplace=True)
kalshi_df

,event_id,market_id,title,descriptive_text,match_key
0,KXKHLGAME-26MAR311000SALAVT,kxkhlgame 26mar311000salavt sal,salavat yulaev ufa vs avtomobilist yekaterinbu...,salavat yulaev ufa vs avtomobilist yekaterinbu...,unknown_2026
1,KXKHLGAME-26MAR311000SALAVT,kxkhlgame 26mar311000salavt avt,salavat yulaev ufa vs avtomobilist yekaterinbu...,salavat yulaev ufa vs avtomobilist yekaterinbu...,unknown_2026
2,KXKHLGAME-26MAR311230SKACSK,kxkhlgame 26mar311230skacsk ska,ska st petersburg vs cska moscow winner,ska st petersburg vs cska moscow winner,moscow_petersburg_2026
3,KXKHLGAME-26MAR311230SKACSK,kxkhlgame 26mar311230skacsk csk,ska st petersburg vs cska moscow winner,ska st petersburg vs cska moscow winner,moscow_petersburg_2026
4,KXATPCHALLENGERMATCH-26MAR29CARLLA,kxatpchallengermatch 26mar29carlla car,pablo carreno busta win the carreno busta vs l...,pablo carreno busta win the carreno busta vs l...,unknown_2026
...,...,...,...,...,...
77,KXMLBHR-26MAR281610BOSCIN,kxmlbhr 26mar281610boscin cinwbenson30 2,benson 2 home runs,benson 2 home runs,home_2026
78,KXMLBHR-26MAR281610BOSCIN,kxmlbhr 26mar281610boscin boswabreu52 1,wilyer abreu 1 home runs,wilyer abreu 1 home runs,home_2026
79,KXMLBHR-26MAR281610BOSCIN,kxmlbhr 26mar281610boscin boswabreu52 2,wilyer abreu 2 home runs,wilyer abreu 2 home runs,home_2026
80,KXMLBHR-26MAR281610BOSCIN,kxmlbhr 26mar281610boscin cinjtrevino35 1,jose trevino 1 home runs,jose trevino 1 home runs,home_jose_2026


In [67]:
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz

# Load once to save memory
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_market_matches(df_kalshi, df_poly, threshold=0.60):
    """
    Ranks matches and returns a DataFrame where the IDs 
    are stored in a single dictionary column for easy hydration.
    """
    # 1. Clean and prepare titles
    k_titles = df_kalshi['descriptive_text'].astype(str).tolist()
    p_titles = df_poly['descriptive_text'].astype(str).tolist()
    
    if not k_titles or not p_titles:
        return pd.DataFrame()

    # 2. Vectorize and compute similarity
    k_embeddings = model.encode(k_titles, convert_to_tensor=True)
    p_embeddings = model.encode(p_titles, convert_to_tensor=True)
    cosine_scores = util.cos_sim(k_embeddings, p_embeddings)

    all_results = []
    for i, k_title in enumerate(k_titles):
        best_match_idx = cosine_scores[i].argmax().item()
        semantic_score = cosine_scores[i][best_match_idx].item()
        
        if semantic_score >= threshold:
            p_match_title = p_titles[best_match_idx]
            
            # Fuzzy secondary check
            fuzzy_val = fuzz.token_sort_ratio(k_title, p_match_title) / 100.0
            
            all_results.append({
                # The ID Dictionary Column
                'platform_ids': {
                    'kalshi': df_kalshi.iloc[i]['event_id'],
                    'poly': df_poly.iloc[best_match_idx]['event_id']
                },
                'kalshi_title': k_title,
                'poly_match': p_match_title,
                'total_score': round((semantic_score + fuzzy_val) / 2, 4),
                'keys': {
                    'kalshi': df_kalshi.iloc[i].get('match_key', 'N/A'),
                    'poly': df_poly.iloc[best_match_idx].get('match_key', 'N/A')
                },
            })

    if not all_results:
        return pd.DataFrame()
        
    # Create DF and sort by best match
    match_df = pd.DataFrame(all_results).sort_values(by='total_score', ascending=False)
    return match_df.reset_index(drop=True)


match_df = get_market_matches(kalshi_df, poly_df)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4659.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [76]:
match_df.to_csv('data/match_df.csv', index=False)
match_df['platform_ids'].values[0]
match_df.values[0]
match_df

,platform_ids,kalshi_title,poly_match,total_score,keys
0,"{'kalshi': 'KXLOLMAP-26MAR29TTWE-2', 'poly': '...",thundertalk gaming win map 2 in the thundertal...,lol team we vs thundertalk gaming game 2 winne...,0.8202,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
1,"{'kalshi': 'KXLOLMAP-26MAR29TTWE-1', 'poly': '...",thundertalk gaming win map 1 in the thundertal...,lol team we vs thundertalk gaming game 1 winne...,0.8167,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
2,"{'kalshi': 'KXLOLMAP-26MAR29TTWE-2', 'poly': '...",team we win map 2 in the thundertalk gaming vs...,lol team we vs thundertalk gaming game 2 winne...,0.7955,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
3,"{'kalshi': 'KXLOLMAP-26MAR29TTWE-1', 'poly': '...",team we win map 1 in the thundertalk gaming vs...,lol team we vs thundertalk gaming game 1 winne...,0.7898,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
4,{'kalshi': 'KXATPCHALLENGERMATCH-26MAR29CARLLA...,pablo carreno busta win the carreno busta vs l...,alicante pablo carreno busta vs pablo llamas r...,0.7576,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
5,{'kalshi': 'KXATPCHALLENGERMATCH-26MAR29CARLLA...,pablo llamas ruiz win the carreno busta vs lla...,alicante pablo carreno busta vs pablo llamas r...,0.7483,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
6,"{'kalshi': 'KXKHLGAME-26MAR311230SKACSK', 'pol...",ska st petersburg vs cska moscow winner,khl ska st petersburg vs cska moscow ska st pe...,0.7394,"{'kalshi': 'moscow_petersburg_2026', 'poly': '..."
7,"{'kalshi': 'KXKHLGAME-26MAR311230SKACSK', 'pol...",ska st petersburg vs cska moscow winner,khl ska st petersburg vs cska moscow ska st pe...,0.7394,"{'kalshi': 'moscow_petersburg_2026', 'poly': '..."
8,"{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'pol...",salavat yulaev ufa vs avtomobilist yekaterinbu...,khl salavat yulaev ufa vs avtomobilist yekater...,0.7094,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
9,"{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'pol...",salavat yulaev ufa vs avtomobilist yekaterinbu...,khl salavat yulaev ufa vs avtomobilist yekater...,0.7094,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
